In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from sympy import *
from sympy.abc import x

from transformers import BertTokenizer, BertTokenizerFast

from deepxube.domains.symbolic_regression import SymbolicState, SymbolicGoal, SymbolicRegression

C:\Users\Cale\miniconda3\envs\deepxube\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tokenizer setup

In [2]:
def normalize(expr):
    import re
    return " ".join(re.findall(r'\d+|[a-zA-Z]+|[\+\-\*/\^\=\(\)]', expr))

In [3]:
tokenizer = BertTokenizer.from_pretrained(
    './', 
    lowercase=True,
)
# tokenizer.vocab

In [4]:
# Why do I need to pass the attention mask in this case? It's 1 for "real" tokens and 0 for 
encoding = tokenizer(
    'sin(x) 1 2 3',
    padding='max_length',
    return_tensors='np',
    max_length=128
)

np.concatenate((encoding['input_ids'], encoding['attention_mask']), axis=None)

array([ 2,  5, 15, 17, 16, 21, 22, 23,  3,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0

## Testing to_np

In [5]:
def _state_to_np(state: SymbolicState, tokenizer: BertTokenizer) -> np.ndarray:
    encoding = tokenizer(
        str(state.expr),
        padding='max_length',
        return_tensors='np',
        max_length=128
    )
    return np.concatenate((encoding['input_ids'], encoding['attention_mask']), axis=None)

def to_np(states: list[SymbolicState], goals: list[SymbolicGoal]) -> list[list[np.ndarray]]:
    # Each row should be a problem instance
    # should each row (i.e. the token array) be padded to the same length?
    tokenizer = BertTokenizer.from_pretrained('./', lowercase=True)
    tokens = [_state_to_np(state, tokenizer) for state in states]
    return [[ts, g.xs, g.ys] for ts, g in zip(tokens, goals)]

In [6]:
start_state = SymbolicState(2*x + x**2)
_state_to_np(start_state, tokenizer)

array([ 2, 17, 11, 11, 22,  9, 22, 11, 17,  3,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0

In [7]:
start_state1 = SymbolicState(2*x + x**2)

goal1 = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 4, 9, 16]),
    tolerance=0
)

start_state2 = SymbolicState(3*x + 1 + sin(-x))

goal2 = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 8, 27, 64]),
    tolerance=0
)

out = to_np(
    [start_state1, start_state2], 
    [goal1, goal2]
)

In [8]:
out

[[array([ 2, 17, 11, 11, 22,  9, 22, 11, 17,  3,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,